In [1]:
import sys
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad

# Adjust this depending on where your notebook lives
sys.path.append(os.path.abspath(".."))  # two levels up
from data_utils.graph_helpers import *

/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_DIR = "../datasets/up_to_date_sep10/"
TARGET_FILE = "finetune_train.h5ad"
# TARGET_FILE = "finetune_test.h5ad"
# TARGET_FILE = "pretrain.h5ad"
adata = ad.read_h5ad(DATA_DIR + TARGET_FILE)

# period taxa are the exceptions to the rule that taxa are only in one period
period_taxa = pd.read_csv(DATA_DIR + "period_taxa.csv", header=None)
period_taxa = period_taxa.drop(index=0)
merged_taxa = period_taxa.apply(lambda row: '.'.join(row.dropna().astype(str)), axis=1)
period_taxa.index = merged_taxa


/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


First, add taxonomy splits

In [20]:
# Define a function to assign categories based on the period_taxa DataFrame
def assign_categories(taxa_name, period_taxa):
    split_taxa = taxa_name.split(".")
    if len(split_taxa) > 6:
        # Use the period_taxa Series to assign categories properly
        if taxa_name not in period_taxa.index:
            raise ValueError(f"Taxa '{taxa_name}' not found in period_taxa.")
        return period_taxa.loc[taxa_name]
    else:
        # Assign generic categories for taxa with 6 or fewer levels
        return split_taxa

# Apply the function to the taxa names and create a new varm
taxon_lists = adata.var["taxa"].apply(assign_categories, period_taxa=period_taxa)
categories = ["Domain", "Phylum", "Class", "Order", "Family", "Genus"]

taxon_df = pd.DataFrame(taxon_lists.tolist(), index=adata.var_names, columns=categories)
adata.varm['taxonomy'] = taxon_df

Now, group by same valid_name (same name after removing invalid parts of taxa, like "Incertae Sedis")

In [21]:
valids_list = {}
for full_name, name_array in taxon_df.iterrows():
    valid_chain, _, _ = get_valid_name(name_array.to_list())
    valids_list[full_name] = tuple(valid_chain)
    
from collections import defaultdict

# Group full_names by identical valid_chain
groups = defaultdict(list)

for full_name, valid_chain in valids_list.items():
    key = valid_chain  # lists are unhashable, tuples are fine as dict keys
    groups[key].append(full_name)

print(f"Found {len(groups)} unique valid_chains.")

Found 4296 unique valid_chains.


In [22]:
adata

AnnData object with n_obs × n_vars = 52266 × 4680
    obs: 'study_id', 'continuous_label', 'categorical_label', 'downstream_task'
    var: 'taxa'
    obsm: 'raw_embedding'
    varm: 'taxonomy'
    layers: 'top_512'

Now, for the ones that have >= 2 together, isolate them

In [23]:
more_than_one_col = {k: v for k, v in groups.items() if len(v) > 1}
print(f"Found {len(more_than_one_col)} valid_chains with more than one associated full_name.")

Found 369 valid_chains with more than one associated full_name.


In [24]:
adata.var_names.get_loc('Bacteria.Bacillota.Bacilli.Lactobacillales.Lactobacillaceae.Weissella')

9

In [25]:
import numpy as np
import pandas as pd
from scipy import sparse

X = adata.X  # keep reference to main data matrix
is_sparse = sparse.issparse(X)

cols_to_drop = []

for valid_chain, full_names in more_than_one_col.items():
    if len(full_names) < 2:
        assert False, "This should not happen."

    # Get column indices of these duplicate vars
    dup_indices = [adata.var_names.get_loc(name) for name in full_names if name in adata.var_names]
    if len(dup_indices) < 2:
        continue

    keep_idx = dup_indices[0]  # we'll keep the first one
    drop_indices = dup_indices[1:]
    cols_to_drop.extend(drop_indices)

    if is_sparse:
        submatrix = X[:, dup_indices].tocsc()
        merged_col = np.array(submatrix.sum(axis=1)).ravel()
        # Replace values in the kept column
        X[:, keep_idx] = merged_col.reshape(-1, 1)
    else:
        merged_col = np.sum(X[:, dup_indices], axis=1)
        X[:, keep_idx] = merged_col

print(f"Will drop {len(set(cols_to_drop))} redundant duplicate columns...")

# ---- Drop redundant columns from adata ----
keep_mask = np.ones(X.shape[1], dtype=bool)
keep_mask[list(set(cols_to_drop))] = False

adata_dropped = ad.AnnData(
    X[:, keep_mask],
    obs=adata.obs.copy(),
    var=adata.var.iloc[keep_mask].copy(),
)

print(f"✅ Final adata shape: {adata_dropped.shape}")


Will drop 384 redundant duplicate columns...
✅ Final adata shape: (52266, 4296)


/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [26]:
SAVE_DIR = "../datasets/remove_redundant_oct24/"
os.makedirs(SAVE_DIR, exist_ok=True)

adata_dropped.write_h5ad(SAVE_DIR + TARGET_FILE)

Now, do the same to the hmc_evo2_embeddings.npy

In [ ]:
hmc_evo2_embs = np.load(DATA_DIR + "hmc_evo2_embeddings.npy")
hmc_evo2_embs.shape

(4680,)

In [35]:
hmc_evo2_embs_dropped = hmc_evo2_embs[keep_mask, :]
np.save(SAVE_DIR + "hmc_evo2_embeddings.npy", hmc_evo2_embs_dropped)